# Spark MLlib Taxi Demand Demo

This notebook is a small ML demo for the NYC taxi demand architecture. It can read the Iceberg ML table for the full project pipeline, or a portable Parquet export for teammates who do not have the Iceberg warehouse.

**Goal:** predict `trip_count` from pickup zone, month, day of week, hour, and weekend flag. The default setup balances Yellow Taxi and HVFHV rows, removes `source_id` from the feature set, and one-hot encodes pickup location so the model does not treat location IDs as continuous numbers.


## Demo Outline

1. Start Spark.
2. Choose Iceberg or portable Parquet input.
3. Load the taxi demand ML dataset.
4. Build balanced train/test sets by source.
5. One-hot encode pickup location and assemble ML features.
6. Tune Linear Regression and Random Forest hyperparameters.
7. Compare RMSE, MAE, and R2 across baseline and tuned models.
8. Compare actual versus predicted trip counts on the 2025 holdout set.
9. Save predictions for the application/report layer.

`DEMO_MODE` is enabled by default because the portable Parquet export has millions of rows. Turn it off only for a final full-data run.


## 1. Start Spark

The project source of truth is the Iceberg table. This cell loads the Iceberg Spark runtime package before Spark starts.

If you are running the notebook from VS Code on your Mac, Docker must expose both HDFS ports:

- NameNode RPC: `localhost:9000`
- DataNode transfer: `localhost:9866`

If Spark was already started with different settings, restart the notebook kernel before running this cell again.

In [ ]:
from pathlib import Path
import zipfile

from pyspark import SparkContext
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor, LinearRegression, RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

RUNNING_IN_DOCKER = Path("/workspace").exists()

# Use "iceberg" for the project pipeline, or "parquet" for a teammate-friendly export.
INPUT_MODE = "parquet"
# INPUT_MODE = "iceberg"

# Keep demo runs fast. Set DEMO_MODE = False for the final full-data run.
DEMO_MODE = True
BALANCE_SOURCES = True
USE_SOURCE_FEATURE = False
RUN_HYPERPARAMETER_TUNING = True
TRAIN_SAMPLE_FRACTION = 0.05 if DEMO_MODE else 1.0
TEST_SAMPLE_FRACTION = 0.10 if DEMO_MODE else 1.0
TUNING_SAMPLE_FRACTION = 0.20 if DEMO_MODE else 0.25
SHUFFLE_PARTITIONS = "8" if DEMO_MODE else "12"
PLOT_SAMPLE_FRACTION = 0.02
PLOT_SAMPLE_LIMIT = 5000

ICEBERG_PACKAGE = "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2"
ICEBERG_WAREHOUSE = (
    "hdfs://namenode:9000/user/data/warehouse"
    if RUNNING_IN_DOCKER
    else "hdfs://localhost:9000/user/data/warehouse"
)
ML_TABLE = "nyc.taxi_demand_ml"
PORTABLE_PARQUET_PATH = (
    "/workspace/output/taxi_demand_ml_parquet"
    if RUNNING_IN_DOCKER
    else "output/taxi_demand_ml_parquet"
)
PORTABLE_PARQUET_ZIP_PATH = f"{PORTABLE_PARQUET_PATH}.zip"

# If this cell is rerun after a failed Spark startup, force Spark to rebuild with the current config.
active_session = SparkSession.getActiveSession()
if active_session is not None:
    active_session.stop()
elif SparkContext._active_spark_context is not None:
    SparkContext._active_spark_context.stop()

spark_builder = (
    SparkSession.builder
    .appName("Spark MLlib Taxi Demand Demo")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTITIONS)
    .config("spark.default.parallelism", SHUFFLE_PARTITIONS)
)

if INPUT_MODE == "iceberg":
    spark_builder = (
        spark_builder
        .config("spark.jars.packages", ICEBERG_PACKAGE)
        .config("spark.sql.catalog.nyc", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.nyc.type", "hadoop")
        .config("spark.sql.catalog.nyc.warehouse", ICEBERG_WAREHOUSE)
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.hadoop.dfs.client.use.datanode.hostname", "false" if RUNNING_IN_DOCKER else "true")
    )

spark = spark_builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Input mode: {INPUT_MODE}")
print(f"Demo mode: {DEMO_MODE}")
print(f"Balance sources: {BALANCE_SOURCES}")
print(f"Use source as feature: {USE_SOURCE_FEATURE}")
print(f"Shuffle partitions: {SHUFFLE_PARTITIONS}")
if INPUT_MODE == "iceberg":
    print(f"Iceberg warehouse: {ICEBERG_WAREHOUSE}")
else:
    print("Iceberg package/config skipped for parquet input")


## Optional: Create a Portable Parquet Export

Run this cell only if you have the Iceberg table and want to share the demo dataset. Send your teammate the exported folder, not random data files from the Iceberg warehouse.

A clean Parquet export has the rows and schema the ML demo needs. The Iceberg warehouse also contains table metadata and manifests, so copying only warehouse parquet files is fragile.

In [ ]:
# Uncomment and run this once to create a portable dataset for teammates.
# (
#     spark.read.table(ML_TABLE)
#     .coalesce(4)
#     .write
#     .mode("overwrite")
#     .parquet(PORTABLE_PARQUET_PATH)
# )
# print(f"Wrote portable Parquet export to {PORTABLE_PARQUET_PATH}")


## 2. Load the ML Dataset

With `INPUT_MODE = "parquet"`, this reads the local portable Parquet export and does not scan the Iceberg table. The notebook still uses Spark MLlib for modeling.


In [ ]:
required_columns = [
    "source_id",
    "source_name",
    "PULocationID",
    "pickup_year",
    "pickup_month",
    "pickup_day_of_week",
    "pickup_hour",
    "is_weekend",
    "trip_count",
]

if INPUT_MODE == "iceberg":
    raw_df = spark.read.table(ML_TABLE)
    input_description = ML_TABLE
elif INPUT_MODE == "parquet":
    parquet_path = Path(PORTABLE_PARQUET_PATH)
    parquet_zip_path = Path(PORTABLE_PARQUET_ZIP_PATH)
    if not parquet_path.exists() and parquet_zip_path.exists():
        print(f"Extracting {parquet_zip_path} to {parquet_path.parent}")
        with zipfile.ZipFile(parquet_zip_path, "r") as archive:
            archive.extractall(parquet_path.parent)
    raw_df = spark.read.parquet(PORTABLE_PARQUET_PATH)
    input_description = PORTABLE_PARQUET_PATH
else:
    raise ValueError('INPUT_MODE must be "iceberg" or "parquet"')

missing_columns = [column for column in required_columns if column not in raw_df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if raw_df.limit(1).count() == 0:
    raise ValueError(f"No rows found in {input_description}")

print(f"Loaded dataset from {input_description}")
print(f"Input partitions: {raw_df.rdd.getNumPartitions()}")
raw_df.select(required_columns).printSchema()
raw_df.select(required_columns).show(5, truncate=False)


## 3. Prepare Modeling Columns

`PULocationID` is kept as a categorical string so Spark can one-hot encode it. `source_name` stays in the dataframe for balancing and diagnostics, but the default feature set does not include `source_id` or source name.


In [ ]:
numeric_feature_cols = [
    "pickup_month",
    "pickup_day_of_week",
    "pickup_hour",
    "is_weekend_int",
]

encoded_feature_cols = ["PULocationID_ohe"]
preprocessing_stages = [
    StringIndexer(
        inputCol="PULocationID_string",
        outputCol="PULocationID_index",
        handleInvalid="keep",
    ),
    OneHotEncoder(
        inputCols=["PULocationID_index"],
        outputCols=["PULocationID_ohe"],
        handleInvalid="keep",
    ),
]

if USE_SOURCE_FEATURE:
    preprocessing_stages.extend([
        StringIndexer(
            inputCol="source_name",
            outputCol="source_name_index",
            handleInvalid="keep",
        ),
        OneHotEncoder(
            inputCols=["source_name_index"],
            outputCols=["source_name_ohe"],
            handleInvalid="keep",
        ),
    ])
    encoded_feature_cols.append("source_name_ohe")

assembler_input_cols = numeric_feature_cols + encoded_feature_cols
preprocessing_stages.append(
    VectorAssembler(
        inputCols=assembler_input_cols,
        outputCol="features",
        handleInvalid="skip",
    )
)
preprocessing_pipeline = Pipeline(stages=preprocessing_stages)

dataset = (
    raw_df
    .select(
        F.col("source_id").cast("int"),
        F.col("source_name").cast("string"),
        F.col("PULocationID").cast("long"),
        F.col("PULocationID").cast("string").alias("PULocationID_string"),
        F.col("pickup_year").cast("int"),
        F.col("pickup_month").cast("int"),
        F.col("pickup_day_of_week").cast("int"),
        F.col("pickup_hour").cast("int"),
        F.col("is_weekend").cast("boolean"),
        F.col("trip_count").cast("double").alias("trip_count"),
    )
    .dropna(subset=[
        "source_name",
        "PULocationID",
        "pickup_year",
        "pickup_month",
        "pickup_day_of_week",
        "pickup_hour",
        "is_weekend",
        "trip_count",
    ])
    .withColumn("is_weekend_int", F.col("is_weekend").cast("int"))
    .withColumn("label", F.col("trip_count"))
    .repartition(int(SHUFFLE_PARTITIONS))
)

print("Numeric feature columns:")
for column in numeric_feature_cols:
    print(f"- {column}")

print("Encoded feature columns:")
for column in encoded_feature_cols:
    print(f"- {column}")


## 4. Balance Sources and Split Train/Test Data

The project framing trains on 2023-2024 and tests on 2025. Because Yellow Taxi and HVFHV have different volumes, this step can undersample the larger source so both sources contribute roughly equally before model training and evaluation.


In [ ]:
def source_counts(df, title):
    rows = df.groupBy("source_name").count().orderBy("source_name").collect()
    total = sum(row["count"] for row in rows)
    print(title)
    for row in rows:
        share = (row["count"] / total * 100.0) if total else 0.0
        print(f"- {row['source_name']}: {row['count']:,} rows ({share:.1f}%)")
    return rows


def balance_by_source(df, title):
    before_rows = source_counts(df, f"{title} before balancing")
    if not BALANCE_SOURCES or len(before_rows) < 2:
        return df

    min_count = min(row["count"] for row in before_rows)
    fractions = {
        row["source_name"]: min(1.0, min_count / row["count"])
        for row in before_rows
        if row["count"] > 0
    }
    print(f"{title} balance fractions: {fractions}")
    balanced_df = df.sampleBy("source_name", fractions, seed=42)
    source_counts(balanced_df, f"{title} after balancing")
    return balanced_df


train_years = [2023, 2024]
test_year = 2025

full_train_df = dataset.filter(F.col("pickup_year").isin(train_years))
full_test_df = dataset.filter(F.col("pickup_year") == test_year)

balanced_train_df = balance_by_source(full_train_df, "Train")
balanced_test_df = balance_by_source(full_test_df, "Test")

if DEMO_MODE:
    train_df = balanced_train_df.sample(False, TRAIN_SAMPLE_FRACTION, seed=42)
    test_df = balanced_test_df.sample(False, TEST_SAMPLE_FRACTION, seed=42)
else:
    train_df = balanced_train_df
    test_df = balanced_test_df

train_df = train_df.persist(StorageLevel.MEMORY_AND_DISK)
test_df = test_df.persist(StorageLevel.MEMORY_AND_DISK)

train_count = train_df.count()
test_count = test_df.count()

if train_count == 0:
    raise ValueError(f"No training rows found for years: {train_years}")
if test_count == 0:
    raise ValueError(f"No test rows found for year: {test_year}")

print(f"Train years: {train_years}")
print(f"Test year: {test_year}")
print(f"Train sample fraction: {TRAIN_SAMPLE_FRACTION}")
print(f"Test sample fraction: {TEST_SAMPLE_FRACTION}")
print(f"Train rows used: {train_count:,}")
print(f"Test rows used: {test_count:,}")

source_counts(train_df, "Train rows used by source")
source_counts(test_df, "Test rows used by source")

feature_columns_to_keep = [
    "source_id",
    "source_name",
    "PULocationID",
    "pickup_year",
    "pickup_month",
    "pickup_day_of_week",
    "pickup_hour",
    "is_weekend",
    "trip_count",
    "label",
    "features",
]

feature_model = preprocessing_pipeline.fit(train_df)
train_features_df = feature_model.transform(train_df).select(feature_columns_to_keep).persist(StorageLevel.MEMORY_AND_DISK)
test_features_df = feature_model.transform(test_df).select(feature_columns_to_keep).persist(StorageLevel.MEMORY_AND_DISK)

train_feature_count = train_features_df.count()
test_feature_count = test_features_df.count()
print(f"Train feature rows: {train_feature_count:,}")
print(f"Test feature rows: {test_feature_count:,}")

train_df.unpersist()
test_df.unpersist()


## 5. Train Baseline Spark MLlib Models

The baseline models train on the prepared `features` column. Location one-hot encoding is fitted once before this step, so each model does not repeatedly refit the same preprocessing pipeline.


In [ ]:
model_specs = [
    (
        "LinearRegression",
        LinearRegression(
            featuresCol="features",
            labelCol="label",
            predictionCol="prediction",
            maxIter=20,
            regParam=0.1,
            elasticNetParam=0.0,
        ),
    ),
    (
        "RandomForestRegressor",
        RandomForestRegressor(
            featuresCol="features",
            labelCol="label",
            predictionCol="prediction",
            numTrees=20,
            maxDepth=7,
            seed=42,
        ),
    ),
    # Uncomment for a heavier comparison run.
    # (
    #     "GBTRegressor",
    #     GBTRegressor(
    #         featuresCol="features",
    #         labelCol="label",
    #         predictionCol="prediction",
    #         maxIter=20,
    #         maxDepth=5,
    #         seed=42,
    #     ),
    # ),
]

metric_names = ["rmse", "mae", "r2"]
evaluators = {
    metric: RegressionEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric,
    )
    for metric in metric_names
}

trained_models = {}
evaluation_rows = []

for model_name, estimator in model_specs:
    print(f"Training {model_name}...")
    model = estimator.fit(train_features_df)
    predictions = model.transform(test_features_df)

    metrics = {
        metric: float(evaluator.evaluate(predictions))
        for metric, evaluator in evaluators.items()
    }
    evaluation_rows.append({"model": model_name, **metrics})
    trained_models[model_name] = model

    print(
        f"{model_name}: "
        f"RMSE={metrics['rmse']:.4f}, "
        f"MAE={metrics['mae']:.4f}, "
        f"R2={metrics['r2']:.4f}"
    )


## 6. Tune Linear Regression and Random Forest

Both baseline model families get a small `TrainValidationSplit` tuning pass. The grids are intentionally small for a class-project demo; widen them only for a final full-data run.


In [ ]:
if RUN_HYPERPARAMETER_TUNING:
    try:
        tuning_train_df.unpersist()
    except NameError:
        pass

    tuning_train_df = train_features_df.sample(False, TUNING_SAMPLE_FRACTION, seed=42).persist(StorageLevel.MEMORY_AND_DISK)
    tuning_train_count = tuning_train_df.count()

    if tuning_train_count == 0:
        tuning_train_df.unpersist()
        tuning_train_df = train_features_df
        tuning_train_count = train_feature_count

    print(f"Tuning sample fraction: {TUNING_SAMPLE_FRACTION}")
    print(f"Tuning sample rows: {tuning_train_count:,}")

    tuning_specs = []

    tuned_lr = LinearRegression(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        maxIter=30,
    )
    lr_param_grid = (
        ParamGridBuilder()
        .addGrid(tuned_lr.regParam, [0.01, 0.1])
        .addGrid(tuned_lr.elasticNetParam, [0.0, 0.5])
        .build()
    )
    tuning_specs.append(("TunedLinearRegression", tuned_lr, lr_param_grid))

    tuned_rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        seed=42,
    )
    rf_param_grid = (
        ParamGridBuilder()
        .addGrid(tuned_rf.numTrees, [20, 40])
        .addGrid(tuned_rf.maxDepth, [6, 8])
        .addGrid(tuned_rf.minInstancesPerNode, [1])
        .build()
    )
    tuning_specs.append(("TunedRandomForestRegressor", tuned_rf, rf_param_grid))

    all_tuning_rows = []

    for tuned_model_name, estimator, param_grid in tuning_specs:
        print(f"Tuning {tuned_model_name} with {len(param_grid)} parameter combinations...")
        evaluation_rows = [row for row in evaluation_rows if row["model"] != tuned_model_name]
        trained_models.pop(tuned_model_name, None)

        tvs = TrainValidationSplit(
            estimator=estimator,
            estimatorParamMaps=param_grid,
            evaluator=evaluators["rmse"],
            trainRatio=0.8,
            parallelism=2,
            seed=42,
        )
        tvs_model = tvs.fit(tuning_train_df)

        tuning_rows = []
        for param_map, validation_rmse in zip(param_grid, tvs_model.validationMetrics):
            params = {param.name: value for param, value in param_map.items()}
            row = {
                "model": tuned_model_name,
                "validation_rmse": float(validation_rmse),
                "params": ", ".join(f"{name}={value}" for name, value in sorted(params.items())),
            }
            tuning_rows.append(row)
            all_tuning_rows.append(row)

        best_tuning_row = min(tuning_rows, key=lambda row: row["validation_rmse"])
        tuned_predictions = tvs_model.bestModel.transform(test_features_df)
        tuned_metrics = {
            metric: float(evaluator.evaluate(tuned_predictions))
            for metric, evaluator in evaluators.items()
        }

        evaluation_rows.append({"model": tuned_model_name, **tuned_metrics})
        trained_models[tuned_model_name] = tvs_model.bestModel

        print(f"Best {tuned_model_name} params: {best_tuning_row['params']}")
        print(
            f"{tuned_model_name}: "
            f"RMSE={tuned_metrics['rmse']:.4f}, "
            f"MAE={tuned_metrics['mae']:.4f}, "
            f"R2={tuned_metrics['r2']:.4f}"
        )

    tuning_results_df = spark.createDataFrame(all_tuning_rows)
    tuning_results_df.orderBy("model", "validation_rmse").show(truncate=False)
else:
    print("Hyperparameter tuning skipped. Set RUN_HYPERPARAMETER_TUNING = True to enable it.")


## 7. Compare Metrics

Lower RMSE and MAE are better. Higher R2 is better. The selected model is the one with the lowest test RMSE on the 2025 holdout set.


In [ ]:
metrics_df = spark.createDataFrame(evaluation_rows)
metrics_df.orderBy("rmse").show(truncate=False)

best_result = min(evaluation_rows, key=lambda row: row["rmse"])
best_model_name = best_result["model"]
best_model = trained_models[best_model_name]

print(f"Best model by RMSE: {best_model_name}")


In [ ]:
print(f"""
Best Model: {best_model_name}
RMSE: {best_result['rmse']:.4f}
MAE: {best_result['mae']:.4f}
R2: {best_result['r2']:.4f}
""")


In [ ]:
import matplotlib.pyplot as plt

metrics_pdf = metrics_df.orderBy("rmse").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(metrics_pdf["model"], metrics_pdf["rmse"], color="#2563eb")
axes[0].invert_yaxis()
axes[0].set_title("Model Comparison by RMSE")
axes[0].set_xlabel("RMSE, lower is better")

axes[1].barh(metrics_pdf["model"], metrics_pdf["mae"], color="#16a34a")
axes[1].invert_yaxis()
axes[1].set_title("Model Comparison by MAE")
axes[1].set_xlabel("MAE, lower is better")

plt.tight_layout()
plt.show()


In [ ]:
Path("output").mkdir(exist_ok=True)
best_model_output_path = "output/best_model_taxi_demand"

best_model.write().overwrite().save(best_model_output_path)
print(f"Best model saved to {best_model_output_path}")


## 8. Actual vs Predicted Comparison

These diagnostics compare the selected model's predictions with the actual 2025 demand counts. The residual is `predicted_trip_count - actual_trip_count`, so positive values mean the model overpredicted demand.


In [ ]:
Path("output").mkdir(exist_ok=True)

try:
    test_predictions.unpersist()
except NameError:
    pass

test_predictions = (
    best_model
    .transform(test_features_df)
    .select(
        "source_name",
        "PULocationID",
        "pickup_year",
        "pickup_month",
        "pickup_day_of_week",
        "pickup_hour",
        "is_weekend",
        F.col("trip_count").alias("actual_trip_count"),
        F.col("prediction").alias("predicted_trip_count"),
    )
    .withColumn("prediction_error", F.col("predicted_trip_count") - F.col("actual_trip_count"))
    .withColumn("absolute_error", F.abs(F.col("prediction_error")))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

test_prediction_count = test_predictions.count()
print(f"2025 prediction rows: {test_prediction_count:,}")

sample_predictions = (
    test_predictions
    .select(
        "source_name",
        "PULocationID",
        "pickup_year",
        "pickup_month",
        "pickup_day_of_week",
        "pickup_hour",
        "is_weekend",
        F.round("actual_trip_count", 2).alias("actual_trip_count"),
        F.round("predicted_trip_count", 2).alias("predicted_trip_count"),
        F.round("prediction_error", 2).alias("prediction_error"),
        F.round("absolute_error", 2).alias("absolute_error"),
    )
    .orderBy("source_name", "PULocationID", "pickup_month", "pickup_day_of_week", "pickup_hour")
)

sample_predictions.show(10, truncate=False)
sample_predictions.limit(1000).toPandas().to_csv("output/sample_predictions.csv", index=False)
print("Sample predictions saved to output/sample_predictions.csv")


In [ ]:
prediction_summary = (
    test_predictions
    .groupBy("source_name")
    .agg(
        F.count("*").alias("rows"),
        F.round(F.avg("actual_trip_count"), 2).alias("avg_actual_trip_count"),
        F.round(F.avg("predicted_trip_count"), 2).alias("avg_predicted_trip_count"),
        F.round(F.avg("prediction_error"), 2).alias("mean_error"),
        F.round(F.avg("absolute_error"), 2).alias("mean_absolute_error"),
        F.round(F.sqrt(F.avg(F.col("prediction_error") * F.col("prediction_error"))), 2).alias("rmse"),
    )
    .orderBy("source_name")
)

prediction_summary.show(truncate=False)

largest_errors = (
    test_predictions
    .select(
        "source_name",
        "PULocationID",
        "pickup_month",
        "pickup_day_of_week",
        "pickup_hour",
        F.round("actual_trip_count", 2).alias("actual_trip_count"),
        F.round("predicted_trip_count", 2).alias("predicted_trip_count"),
        F.round("prediction_error", 2).alias("prediction_error"),
        F.round("absolute_error", 2).alias("absolute_error"),
    )
    .orderBy(F.desc("absolute_error"))
)

largest_errors.show(10, truncate=False)


In [ ]:
comparison_sample_pdf = (
    test_predictions
    .sample(False, PLOT_SAMPLE_FRACTION, seed=42)
    .limit(PLOT_SAMPLE_LIMIT)
    .select("source_name", "actual_trip_count", "predicted_trip_count")
    .toPandas()
)

if comparison_sample_pdf.empty:
    comparison_sample_pdf = (
        test_predictions
        .limit(PLOT_SAMPLE_LIMIT)
        .select("source_name", "actual_trip_count", "predicted_trip_count")
        .toPandas()
    )

if comparison_sample_pdf.empty:
    print("No rows available for actual vs predicted plot.")
else:
    fig, ax = plt.subplots(figsize=(7, 7))
    for source_name, group in comparison_sample_pdf.groupby("source_name"):
        ax.scatter(
            group["actual_trip_count"],
            group["predicted_trip_count"],
            alpha=0.25,
            s=12,
            label=source_name,
        )

    max_axis = max(
        comparison_sample_pdf["actual_trip_count"].max(),
        comparison_sample_pdf["predicted_trip_count"].max(),
        1,
    )
    ax.plot([0, max_axis], [0, max_axis], color="#dc2626", linestyle="--", linewidth=1, label="perfect prediction")
    ax.set_title("Actual vs Predicted 2025 Trip Count")
    ax.set_xlabel("Actual trip count")
    ax.set_ylabel("Predicted trip count")
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
hourly_comparison_pdf = (
    test_predictions
    .groupBy("source_name", "pickup_hour")
    .agg(
        F.avg("actual_trip_count").alias("avg_actual_trip_count"),
        F.avg("predicted_trip_count").alias("avg_predicted_trip_count"),
        F.avg("prediction_error").alias("avg_prediction_error"),
    )
    .orderBy("source_name", "pickup_hour")
    .toPandas()
)

if hourly_comparison_pdf.empty:
    print("No rows available for hourly actual vs predicted plot.")
else:
    sources = list(hourly_comparison_pdf["source_name"].drop_duplicates())
    fig, axes = plt.subplots(len(sources), 1, figsize=(10, 4 * len(sources)), sharex=True)
    if len(sources) == 1:
        axes = [axes]

    for ax, source_name in zip(axes, sources):
        group = hourly_comparison_pdf[hourly_comparison_pdf["source_name"] == source_name]
        ax.plot(group["pickup_hour"], group["avg_actual_trip_count"], marker="o", label="Actual")
        ax.plot(group["pickup_hour"], group["avg_predicted_trip_count"], marker="o", label="Predicted")
        ax.set_title(f"Average Actual vs Predicted Demand by Hour: {source_name}")
        ax.set_ylabel("Average trip count")
        ax.set_xticks(range(0, 24, 2))
        ax.legend()

    axes[-1].set_xlabel("Pickup hour")
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 4))
    for source_name in sources:
        group = hourly_comparison_pdf[hourly_comparison_pdf["source_name"] == source_name]
        ax.plot(group["pickup_hour"], group["avg_prediction_error"], marker="o", label=source_name)
    ax.axhline(0, color="#111827", linewidth=1)
    ax.set_title("Average Prediction Error by Pickup Hour")
    ax.set_xlabel("Pickup hour")
    ax.set_ylabel("Predicted minus actual trips")
    ax.set_xticks(range(0, 24, 2))
    ax.legend()
    plt.tight_layout()
    plt.show()

residual_pdf = (
    test_predictions
    .sample(False, PLOT_SAMPLE_FRACTION, seed=24)
    .limit(PLOT_SAMPLE_LIMIT)
    .select("source_name", "prediction_error")
    .toPandas()
)

if not residual_pdf.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    for source_name, group in residual_pdf.groupby("source_name"):
        ax.hist(group["prediction_error"], bins=40, alpha=0.45, label=source_name)
    ax.axvline(0, color="#111827", linewidth=1)
    ax.set_title("Residual Distribution on 2025 Holdout")
    ax.set_xlabel("Predicted minus actual trips")
    ax.set_ylabel("Frequency")
    ax.legend()
    plt.tight_layout()
    plt.show()


## 9. Save and Review Prediction Output

In demo mode this saves the sampled 2025 holdout predictions. With `DEMO_MODE = False`, it transforms and scores the full dataset for downstream reporting or app work.


In [ ]:
if DEMO_MODE:
    prediction_output_base = test_features_df
    prediction_output_description = "sampled 2025 holdout rows"
else:
    prediction_output_base = feature_model.transform(dataset).select(feature_columns_to_keep)
    prediction_output_description = "full dataset"

full_predictions = (
    best_model
    .transform(prediction_output_base)
    .select(
        "source_name",
        "PULocationID",
        "pickup_year",
        "pickup_month",
        "pickup_day_of_week",
        "pickup_hour",
        "is_weekend",
        F.col("trip_count").alias("actual_trip_count"),
        F.col("prediction").alias("predicted_trip_count"),
    )
    .withColumn("prediction_error", F.col("predicted_trip_count") - F.col("actual_trip_count"))
    .withColumn("absolute_error", F.abs(F.col("prediction_error")))
)

print(f"Prediction output base: {prediction_output_description}")
full_predictions.select(
    "source_name",
    "PULocationID",
    "pickup_year",
    "pickup_month",
    "pickup_day_of_week",
    "pickup_hour",
    "is_weekend",
    F.round("actual_trip_count", 2).alias("actual_trip_count"),
    F.round("predicted_trip_count", 2).alias("predicted_trip_count"),
    F.round("prediction_error", 2).alias("prediction_error"),
    F.round("absolute_error", 2).alias("absolute_error"),
).show(10, truncate=False)


In [ ]:
# Predicted trip count distribution.
prediction_distribution_pdf = (
    full_predictions
    .select("predicted_trip_count")
    .sample(False, 0.02, seed=42)
    .limit(5000)
    .toPandas()
)

if prediction_distribution_pdf.empty:
    prediction_distribution_pdf = full_predictions.select("predicted_trip_count").limit(5000).toPandas()

plt.figure(figsize=(8, 5))
plt.hist(prediction_distribution_pdf["predicted_trip_count"], bins=50)
plt.title("Distribution of Predicted Trip Counts")
plt.xlabel("Predicted trip count")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# Demand levels for downstream app/report summaries.
full_predictions = full_predictions.withColumn(
    "demand_level",
    F.when(F.col("predicted_trip_count") < 20, "low")
     .when(F.col("predicted_trip_count") < 50, "medium")
     .when(F.col("predicted_trip_count") < 100, "high")
     .otherwise("very_high")
)


In [ ]:
full_predictions.show(10, truncate=False)


In [ ]:
# Visualize average actual and predicted demand by hour for the saved prediction scope.
hourly_pdf = (
    full_predictions
    .groupBy("pickup_hour")
    .agg(
        F.avg("actual_trip_count").alias("avg_actual_trip_count"),
        F.avg("predicted_trip_count").alias("avg_predicted_trip_count"),
    )
    .orderBy("pickup_hour")
    .toPandas()
)

plt.figure(figsize=(9, 5))
plt.plot(hourly_pdf["pickup_hour"], hourly_pdf["avg_actual_trip_count"], marker="o", label="Actual")
plt.plot(hourly_pdf["pickup_hour"], hourly_pdf["avg_predicted_trip_count"], marker="o", label="Predicted")
plt.title("Average Actual vs Predicted Demand by Hour")
plt.xlabel("Hour of day")
plt.ylabel("Average trip count")
plt.xticks(range(0, 24, 2))
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
Path("output").mkdir(exist_ok=True)
final_predictions_path = "output/final_predictions"

full_predictions.write.mode("overwrite").parquet(final_predictions_path)
print(f"Final predictions saved to {final_predictions_path}")
print(f"Saved prediction scope: {prediction_output_description}")


## 10. Stop Spark

Stop the Spark session when the demo is finished so local resources are released.


In [ ]:
for cached_name in ["tuning_train_df", "train_features_df", "test_features_df", "test_predictions"]:
    cached_df = globals().get(cached_name)
    if cached_df is not None:
        cached_df.unpersist()

spark.stop()
